# Assignment 2 - Image Captioning using CNNs and LSTMs

For this assignemnt, you will be given full-colour images as input data, and attempt to develop an Encoder-Decoder model to create worded captions for the image. For instance, for the image 

<img src='https://towardsdatascience.com/wp-content/uploads/2022/06/0HxBDFFqwpjShEnyb-scaled.jpg' width='250px'/>

Your model may generate something like "A dog running in the sea"


We will work with the Flickr8k dataset, which contains 8000 images, each associated with 5 sample captions.

## Flickr8k Dataset

This dataset contains approximately 1GB of data. Therefore it is strongly reccommended to use the UCC Jupyter server for this assignment.

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

In [ ]:
!mkdir -p data/flickr8k/
!wget "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip" -O "data/flickr8k/Flickr8k_Dataset.zip"
!wget "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip" -O "data/flickr8k/Flickr8k_text.zip"

In [ ]:
%%bash
if [ ! -d "data/flickr8k/Flicker8k_Dataset" ]
then
    unzip "data/flickr8k/Flickr8k_Dataset.zip" -d data/flickr8k/
fi

if [ ! -d "data/flickr8k/Flickr8k_text" ]
then
    unzip "data/flickr8k/Flickr8k_text.zip" -d data/flickr8k/Flickr8k_text
    rm -r "data/flickr8k/Flickr8k_text/__MACOSX"
fi

if [ -d "data/flickr8k/__MACOSX" ]
then
    rm -r "data/flickr8k/__MACOSX"
fi
mkdir -p saved_models

## Data Loading

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import os
import numpy as np

from collections import Counter

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader,Dataset
import torchvision.transforms as T

from PIL import Image


You may ned to use `%cd __wherever__` here

In [ ]:
data_location =  "data/flickr8k"

caption_file = data_location + '/Flickr8k_text/Flickr8k.token.txt'
df = pd.read_csv(caption_file, delimiter='\t', header=None)

# Cleanup filenames
df.iloc[:,0] = df[0].apply(lambda x: x.split('#')[0])
df = df[~df[0].apply(lambda x: '.jpg.1' in x)]

print("There are {} unique captions".format(len(df)))


In [ ]:
data_idx = 42

image_path = data_location+"/Flicker8k_Dataset/"+df.iloc[data_idx,0]
img=mpimg.imread(image_path)
plt.imshow(img)
plt.show()

all_captions=df[1][df[0]==df.iloc[data_idx,0]]

for cap in all_captions:
    print("Caption:",cap)

## Creating PyTorch Dataset

### Tokenization

The first step is to splitthe captions into uniformly formatted lists of 'tokens', which act as the inputs to our models. Tokens can be words or punctuation, which are used similarly as how we used characters in the previous lab.

In order to clean up the text, we will use the Python SpaCY NLP module

In [ ]:
%pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

In [ ]:
spacy_eng = spacy.load("en_core_web_sm")

We will use this to split sentences into "tokens", which we will pass as inputs to our models.

In [ ]:
text = "This is a good place - to find a city!"
[token.text.lower() for token in spacy_eng.tokenizer(text)]

### Vocabulary

We will map every possible 'token' that we encounter into a distinct integer.

In [ ]:
class Vocabulary:
    def __init__(self,freq_threshold):
        #setting the pre-reserved tokens int to string tokens
        self.itos = {0:"<PAD>",1:"<SOS>",2:"<EOS>",3:"<UNK>"}
        
        #string to int tokens
        #its reverse dict self.itos
        self.stoi = {v:k for k,v in self.itos.items()}
        
        self.freq_threshold = freq_threshold
        
    def __len__(self): return len(self.itos)
    
    @staticmethod
    def tokenize(text):
        return [token.text.lower() for token in spacy_eng.tokenizer(text)]
    
    def build_vocab(self, sentence_list):
        frequencies = Counter()
        idx = 4
        
        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1
                
                #add the word to the vocab if it reaches minum frequecy threshold
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
    
    def numericalize(self,text):
        """ For each word in the text corresponding index token for that word form the vocab built as list """
        tokenized_text = self.tokenize(text)
        return [ self.stoi[token] if token in self.stoi else self.stoi["<UNK>"] for token in tokenized_text ] 

In [ ]:
v = Vocabulary(freq_threshold=1)

v.build_vocab(["This is a good place - to find a city!"])
print(v.stoi)
print(v.numericalize("This is a good place ; to find a city here!"))

Notice how ';' and 'here' are unknown tokens, which our volcabularly maps to \<UNK\> -- unkown. \<SOS\> and \<EOS\> Correpsond to Start and End of Sentence

### Dataset

There are ~40000 samples. We will use 30000 samples for training, and the remainder for validation.

In [ ]:
class FlickrDataset(Dataset):
    """
    FlickrDataset
    """
    def __init__(self,root_dir,captions_file,transform=None,train=True,freq_threshold=5):
        self.root_dir = root_dir
        self.df = pd.read_csv(caption_file, delimiter='\t', header=None)
        # Cleanup filenames
        np.random.seed(42)
        self.df = self.df.sample(frac=1)
        
        self.df =  self.df.reset_index(drop=True)
        self.df.iloc[:,0] = self.df[0].apply(lambda x: x.split('#')[0])
        self.df = self.df[~self.df[0].apply(lambda x: '.jpg.1' in x)]
        self.df =self.df.reset_index()
        self.transform = transform
        
        #Get image and caption colum from the dataframe
        self.imgs = self.df[0]
        self.captions = self.df[1]
        
        #Initialize vocabulary and build vocab
        self.vocab = Vocabulary(freq_threshold)
        self.vocab.build_vocab(self.captions.tolist())
        
        if train:
            self.df = self.df.iloc[:30000,:]
        else:
            self.df = self.df.iloc[30000:,:]
        
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        caption = self.captions[idx]
        img_name = self.imgs[idx]
        img_location = os.path.join(self.root_dir,img_name)
        img = Image.open(img_location).convert("RGB")
        
        #apply the transfromation to the image
        if self.transform is not None:
            img = self.transform(img)
        
        #numericalize the caption text
        caption_vec = []
        caption_vec += [self.vocab.stoi["<SOS>"]]
        caption_vec += self.vocab.numericalize(caption)
        caption_vec += [self.vocab.stoi["<EOS>"]]
        
        return img, torch.tensor(caption_vec)

In [ ]:
'''def get_mean_std(loader):
    # Sum of pixels and sum of squared pixels for each channel
    channels_sum, channels_squared_sum, num_batches = 0, 0, 0
    
    for data, _ in loader:
        # data shape: [batch_size, 3, height, width]
        channels_sum += torch.mean(data, dim=[0, 2, 3])
        channels_squared_sum += torch.mean(data**2, dim=[0, 2, 3])
        num_batches += 1
    
    mean = channels_sum / num_batches
    std = (channels_squared_sum / num_batches - mean**2)**0.5
    return mean, std

get_mean_std(loader)
'''

In [ ]:
# Downscale the images, convert to Tensor
transforms = T.Compose([
    T.Resize((224,224)),
    # These are the manually computed means and stdevs in each dimension
    T.ToTensor(),
    T.Normalize((0.485, 0.4461, 0.4039), (0.2695, 0.2623, 0.2772)),
    
])

In [ ]:
train_dataset =  FlickrDataset(
    root_dir = data_location+"/Flicker8k_Dataset",
    captions_file = data_location+"/Flickr8k_text/Flickr8k.token.txt",
    transform=transforms
)

val_dataset =  FlickrDataset(
    root_dir = data_location+"/Flicker8k_Dataset",
    captions_file = data_location+"/Flickr8k_text/Flickr8k.token.txt",
    train=False,
    transform=transforms
)

In [ ]:
def show_image(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001) 

In [ ]:
img, caps = train_dataset[0]
show_image(img,"Image")
print("Token:",caps)
print("Sentence:")
print([train_dataset.vocab.itos[token] for token in caps.tolist()])


### Dataloader

In oder to enable batching for our dataloader, we must pad our captions with dummy tokens to be of uniform length:

In [ ]:
class PadCaptions:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        targets = [item[1] for item in batch]
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets
    
pad_idx = train_dataset.vocab.stoi["<PAD>"]

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    collate_fn = PadCaptions(pad_idx=pad_idx)
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    collate_fn = PadCaptions(pad_idx=pad_idx)
)

# Model Fitting

We will fit an Encoder-Decoder model, with a CNN as the encoder, and a LSTM as the Decoder.

<img src='https://towardsdatascience.com/wp-content/uploads/2022/06/1MqsV98FA4XA99QFs7BXldg.jpeg' width='500px'/>

The key thing to understand is the link bewteen the CNN output, and the RNN input:

<img src='https://learnopencv.com/wp-content/uploads/2024/12/encoder-decoder.png' />

Key:

# Task 1 CNN Encoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        # Layer 1: Input (3, 224, 224) -> Output (32, 112, 112)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Layer 2: Output (64, 56, 56)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Layer 3: Output (128, 28, 28)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # Fully connected layer to reach embed_size
        # Assuming input image is 224x224, after 3 pools it is 28x28
        self.fc = nn.Linear(128 * 28 * 28, embed_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):
        x = self.pool1(F.relu(self.conv1(images)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        
        # Flatten the features
        x = x.view(x.size(0), -1) 
        x = self.dropout(F.relu(self.fc(x)))
        return x

# Task 2 LSTM Decoder

`self.embedding = nn.Embedding(vocab_size, embed_size)`

Is an important component here.

Recall that last week, we created 1-hot vectors (of some length) to represent indivdual characters.

Here, we create a traianable layer that abstracts away this component. This takes in the number of possible tokens in our vocabulary, and outputs a fixed size (not necessarily 1-hot) vector of size embed_size.




In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    # Features is passed from the CNN layer!
    def forward(self, features, captions):
        # Remove the <end> token from the caption for training
        embeddings = self.embedding(captions[:, :-1])
        
        # Add the image features as the first 'word' of the sequence
        # features: (batch, embed_size) -> (batch, 1, embed_size)
        # Combine the CNN output, and the word embeddings, these become the inputs!
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)
        
        hiddens, _ = self.lstm(embeddings)
        outputs = self.linear(hiddens)
        return outputs

### Defining the Full Model:


In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
EMBED_SIZE = 256
HIDDEN_SIZE = 51
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab=train_dataset.vocab
VOCAB_SIZE = len(vocab)

encoder = EncoderCNN(EMBED_SIZE).to(DEVICE)
decoder = DecoderRNN(EMBED_SIZE, HIDDEN_SIZE, VOCAB_SIZE).to(DEVICE)



In [ ]:
from tqdm import tqdm # For a nice progress bar


# Initialize Loss and Optimizer
# We ignore the <PAD> token in loss calculation so it doesn't penalize the model
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])
params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = torch.optim.Adam(params, lr=LEARNING_RATE)

# Move models to device
encoder.to(DEVICE)
decoder.to(DEVICE)

for epoch in range(1, NUM_EPOCHS + 1):
    # --- TRAINING PHASE ---
    encoder.train()
    decoder.train()
    train_loss = 0
    
    # Progress bar for the training loader
    loop = tqdm(train_loader, total=len(train_loader), leave=True)
    for images, captions in loop:
        images, captions = images.to(DEVICE), captions.to(DEVICE)

        # Forward pass
        optimizer.zero_grad()
        
        # Extract features from CNN and pass to RNN
        features = encoder(images)
        outputs = decoder(features, captions)

        # Calculate loss: 
        # outputs shape: (batch, seq_len, vocab_size)
        # targets shape: (batch, seq_len)
        # We flatten them to (batch * seq_len, vocab_size) for CrossEntropy
        loss = criterion(outputs.view(-1, VOCAB_SIZE), captions.view(-1))
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        loop.set_description(f"Epoch [{epoch}/{NUM_EPOCHS}]")
        loop.set_postfix(loss=loss.item())

    # --- VALIDATION PHASE ---
    encoder.eval()
    decoder.eval()
    val_loss = 0
    
    with torch.no_grad():
        for images, captions in val_loader:
            images, captions = images.to(DEVICE), captions.to(DEVICE)
            
            features = encoder(images)
            outputs = decoder(features, captions)
            
            loss = criterion(outputs.view(-1, VOCAB_SIZE), captions.view(-1))
            val_loss += loss.item()

    # Log statistics
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"\n=> Epoch {epoch} Complete")
    print(f"   Avg Train Loss: {avg_train_loss:.4f}")
    print(f"   Avg Val Loss:   {avg_val_loss:.4f}")

### Evaluate on a Sample Image

In [ ]:
def generate_caption(encoder, decoder, image, vocab, max_length=20):
    result_caption = []
    
    with torch.no_grad():
        # image shape: [3, 224, 224] -> [1, 3, 224, 224]
        x = image.unsqueeze(0).to(DEVICE)
        
        # Get image features
        features = encoder(x) # [1, embed_size]
        
        # Start with the <SOS> token
        # We start by passing the image features as the first 'token'
        # The LSTM expects (batch, seq, features). 
        # We will use the features directly to get the first hidden state
        # or use the SOS token embedding.
        
        # Method: Start with image features, then SOS token
        states = None
        # Passing image features as the first sequence element
        inputs = features.unsqueeze(1) # [1, 1, embed_size]
        
        for _ in range(max_length):
            hiddens, states = decoder.lstm(inputs, states)
            output = decoder.linear(hiddens.squeeze(1)) # [1, vocab_size]
            predicted = output.argmax(1)
            
            token = vocab.itos[predicted.item()]
            
            if token == "<EOS>":
                break
                
            result_caption.append(token)
            
            # Prepare next input: embedding of the word we just predicted
            inputs = decoder.embedding(predicted).unsqueeze(1)
            
    return ' '.join(result_caption)

In [ ]:
# 1. Switch models to evaluation mode
encoder.eval()
decoder.eval()

# 2. Get a random batch from validation set
images, captions = next(iter(val_loader))

# 3. Pick the first image in that batch
sample_img = images[0]
sample_cap = captions[0]

# 4. Generate prediction
prediction = generate_caption(encoder, decoder, sample_img, vocab)

# 5. Convert ground truth indices back to words (skipping <PAD>, <SOS>, <EOS>)
ground_truth = [vocab.itos[idx.item()] for idx in sample_cap 
                if vocab.itos[idx.item()] not in ["<PAD>", "<SOS>", "<EOS>"]]
ground_truth = ' '.join(ground_truth)

# 6. Plotting
# Un-normalize the image for display
img_display = sample_img.permute(1, 2, 0).numpy()
# Use the dataset mean/std calculated earlier or ImageNet defaults
mean = np.array([0.485, 0.4461, 0.4039])
std = np.array([0.2695, 0.2623, 0.2772])
img_display = std * img_display + mean
img_display = np.clip(img_display, 0, 1)

plt.imshow(img_display)
plt.title(f"Predicted: {prediction}\nActual: {ground_truth}", fontsize=10)
plt.axis('off')
plt.show()

## Exercises


1. Create a python string entitles "task_1_explanation" that should contain your discussion of the following:
    a. *Impact of Feature Dimensions* Consider `embed_size` and `hidden_size`
    b. Impact of number of CNN complexity. You may vary filters and/layers as you see appropiate
2. Add regularization -- L1L2, Dropout and/or BatchNorm. Discuss the effect, and your chosen model in the string "task_2_explanation"
3. Add clipping by norm to the LSTM parameters. Does this have any effect on the result, and why? Discuss in the string "task_3_explanation"
4. Experiment with a few Batch sizes, and discuss the effect on trianing and results in "task_4_explanation"
